<a href="https://colab.research.google.com/github/mervefilizbaker1/DSC-525-IS-Big-Data-Analytics/blob/main/HW1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 1: Setup and Load

In [1]:
# I import the SparkSession then I created a Spark Session
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Titanic").getOrCreate()

In [2]:
# I loaded the dataset into a Pyspark DataFrame
!wget https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv
df = spark.read.csv("titanic.csv", header=True, inferSchema=True)

--2026-07-14 17:46:43--  https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 60302 (59K) [text/plain]
Saving to: ‘titanic.csv’

titanic.csv         100%[===================>]  58.89K  --.-KB/s    in 0.007s  

2026-07-14 17:46:43 (8.33 MB/s) - ‘titanic.csv’ saved [60302/60302]



In [3]:
# I showed the first 5 rows and printed the schema of the dataset
df.show(5)
df.printSchema()

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
+-----------+--------+------+--------------------+------+----+-----+-----+------

# Part 2: Save as a Table

In [4]:
# I saved the Titanic dataset as a Spark table named 'titanic'
df.write.mode("overwrite").saveAsTable("titanic")

In [6]:
# I showed the first 5 rows of the titanic table and verified that the table was created succesfully
spark.sql("SELECT * FROM titanic").show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
+-----------+--------+------+--------------------+------+----+-----+-----+------

# Part 3: Data Analysis with DataFrame API

In [7]:
# I showed all female passengers in 1st class using DF API
from pyspark.sql.functions import col
df.filter((col("Sex") == "female") & (col("Pclass") == 1)).show()

+-----------+--------+------+--------------------+------+----+-----+-----+--------+--------+-----------+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|  Ticket|    Fare|      Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+--------+--------+-----------+--------+
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|PC 17599| 71.2833|        C85|       C|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|  113803|    53.1|       C123|       S|
|         12|       1|     1|Bonnell, Miss. El...|female|58.0|    0|    0|  113783|   26.55|       C103|       S|
|         32|       1|     1|Spencer, Mrs. Wil...|female|NULL|    1|    0|PC 17569|146.5208|        B78|       C|
|         53|       1|     1|Harper, Mrs. Henr...|female|49.0|    1|    0|PC 17572| 76.7292|        D33|       C|
|         62|       1|     1| Icard, Miss. Amelie|female|38.0|    0|    0|  113572|    8

In [8]:
# I found the average age of survivors using DF API
from pyspark.sql.functions import avg
df.filter(col("Survived") == 1).select(avg("Age")).show()

+------------------+
|          avg(Age)|
+------------------+
|28.343689655172415|
+------------------+



In [9]:
# I printed how many passengers have missing Age values using DF API
missing_age_count= df.filter(col("Age").isNull()).count()
print(f"Number of passengers with missing age: {missing_age_count}")

Number of passengers with missing age: 177


In [11]:
# I showed the survival rate by sex using DF API
from pyspark.sql.functions import sum, count, round
survival_rate_by_sex = df.groupBy("Sex").agg(round((sum("Survived") * 100.0 / count("*")), 2).alias("Survival_Rate"))
survival_rate_by_sex.show()

+------+-------------+
|   Sex|Survival_Rate|
+------+-------------+
|female|         74.2|
|  male|        18.89|
+------+-------------+



# Part 4: Data Analysis with Spark SQL

In [13]:
# I selected the top 10 oldest passengers using Spark SQL
spark.sql("""
SELECT *
FROM titanic
WHERE Age IS NOT NULL
ORDER BY Age DESC
LIMIT 10
""").show()

+-----------+--------+------+--------------------+----+----+-----+-----+----------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name| Sex| Age|SibSp|Parch|    Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+----+----+-----+-----+----------+-------+-----+--------+
|        631|       1|     1|Barkworth, Mr. Al...|male|80.0|    0|    0|     27042|   30.0|  A23|       S|
|        852|       0|     3| Svensson, Mr. Johan|male|74.0|    0|    0|    347060|  7.775| NULL|       S|
|         97|       0|     1|Goldschmidt, Mr. ...|male|71.0|    0|    0|  PC 17754|34.6542|   A5|       C|
|        494|       0|     1|Artagaveytia, Mr....|male|71.0|    0|    0|  PC 17609|49.5042| NULL|       C|
|        117|       0|     3|Connors, Mr. Patrick|male|70.5|    0|    0|    370369|   7.75| NULL|       Q|
|        673|       0|     2|Mitchell, Mr. Hen...|male|70.0|    0|    0|C.A. 24580|   10.5| NULL|       S|
|        746|       0|     1|Crosby, 

In [14]:
# I showed the survival rate by passenger class (Pclass) using Spark SQL
spark.sql("""
SELECT Pclass , ROUND(Sum(Survived)*100.0/ Count(*),2) AS Survival_Rate
FROM titanic
GROUP BY Pclass
ORDER BY Pclass
""").show()

+------+-------------+
|Pclass|Survival_Rate|
+------+-------------+
|     1|        62.96|
|     2|        47.28|
|     3|        24.24|
+------+-------------+



In [15]:
# I showed the maximum fare paid for each embarkation port (Embarked) and excluded the null
spark.sql("""
SELECT Embarked , MAX(Fare) AS Max_Fare
FROM titanic
WHERE Embarked IS NOT NULL
GROUP BY Embarked
ORDER BY Embarked
""").show()

+--------+--------+
|Embarked|Max_Fare|
+--------+--------+
|       C|512.3292|
|       Q|    90.0|
|       S|   263.0|
+--------+--------+



In [16]:
# I showed the average fare of passengers younger than 18 using Spark SQL
spark.sql(
"""
SELECT AVG(Fare) as Average_Fare_Younger_Than_18
FROM titanic
WHERE Age < 18
""").show()

+----------------------------+
|Average_Fare_Younger_Than_18|
+----------------------------+
|           31.22079823008851|
+----------------------------+

